__UPDATE!__

For $\alpha$ runs:

- Load the sophronia/dorothea reconstruced data by IC.
- Apply the selecting criteria of your preference.
- Store all the dataframes for future analysis.

In [1]:
import sys
sys.path.append('/lhome/ific/c/ccortesp/Analysis')

from libs import crudo

import os
import pandas as pd
import pickle

%load_ext autoreload
%autoreload 2

Crudo package loaded successfully.
Available sub-modules: data_management (dm), energy_functions (ef), fit_functions (ff), plotting_tools (pt), topology_functions (tf), utilities (ut).


# Preliminary

In [2]:
# α runs!
RUNS_INFO = [    
                # # --- Cold-Getter: Jan 17th --- #
                # {"run_number": 14714, "duration": 64574, "OK": 1909287, "LOST": 1728359, "real_rate": 56.333},
                # # {"run_number": 14715, "duration": 84365, "OK": 2469062, "LOST": 2239303, "real_rate": 55.809},      # Waveforms in magnetic tape, not processed
                # {"run_number": 14716, "duration": 17036, "OK": 495769 , "LOST": 451306 , "real_rate": 55.592},
                # {"run_number": 14720, "duration": 48518, "OK": 1432110, "LOST": 1294941, "real_rate": 56.207},      # Estimated from fit
                # {"run_number": 14733, "duration": 53881, "OK": 1587637, "LOST": 1429629, "real_rate": 55.998},
                # {"run_number": 14735, "duration": 84987, "OK": 2508569, "LOST": 2267982, "real_rate": 56.203},
                # {"run_number": 14737, "duration": 72705, "OK": 2153786, "LOST": 1960347, "real_rate": 56.586},
                {"run_number": 14739, "duration": 87138, "OK": 2576630, "LOST": 2334343, "real_rate": 56.358},
                # {"run_number": 14741, "duration": 87755, "OK": 2592615, "LOST": 2349583, "real_rate": 56.318},
                # {"run_number": 14743, "duration": 82332, "OK": 2437984, "LOST": 2220643, "real_rate": 56.583},
                # {"run_number": 14745, "duration": 60990, "OK": 1803415, "LOST": 1636317, "real_rate": 56.398},

                # # --- Hot-Getter: Jan 27th --- #
                # {"run_number": 14753, "duration": 86308, "OK": 2188547, "LOST": 1532351, "real_rate": 43.111},
                # {"run_number": 14765, "duration": 55978, "OK": 1259832, "LOST": 735202 , "real_rate": 35.639},
                # {"run_number": 14776, "duration": 47397, "OK": 957081 , "LOST": 481810 , "real_rate": 30.358},        # Data removed
                # {"run_number": 14780, "duration": 88503, "OK": 1600797, "LOST": 682246 , "real_rate": 25.796},
                # # Feb 2025
                # {"run_number": 14782, "duration": 85654, "OK": 1382153, "LOST": 506234 , "real_rate": 22.046},
                # {"run_number": 14784, "duration": 57290, "OK": 842541 , "LOST": 273581 , "real_rate": 19.481},
                # {"run_number": 14789, "duration": 74148, "OK": 976360 , "LOST": 277504 , "real_rate": 16.910},
                # # --- Zero Suppression --- #
                # {"run_number": 14803, "duration": 71721, "OK": 933727 , "LOST": 104677 , "real_rate": 14.478},
                # {"run_number": 14804, "duration": 65116, "OK": 680552 , "LOST": 156995 , "real_rate": 12.862},
                # # --- NO Zero Suppression --- #
                # {"run_number": 14811, "duration": 84651, "OK": 782816 , "LOST": 147773 , "real_rate": 10.993},
                # {"run_number": 14814, "duration": 6344 , "OK": 55679  , "LOST": 10155  , "real_rate": 10.377},
                # {"run_number": 14815, "duration": 86232, "OK": 717786 , "LOST": 121527 , "real_rate": 9.733 },
                # {"run_number": 14816, "duration": 86580, "OK": 659265 , "LOST": 101374 , "real_rate": 8.785 },
                # {"run_number": 14817, "duration": 49662, "OK": 352203 , "LOST": 50474  , "real_rate": 8.108 },
                # {"run_number": 14828, "duration": 53609, "OK": 300827 , "LOST": 100405 , "real_rate": 7.484 },
                # {"run_number": 14829, "duration": 73042, "OK": 387003 , "LOST": 121249 , "real_rate": 6.958 },
                # {"run_number": 14834, "duration": 5779 , "OK": 32864  , "LOST": 3887   , "real_rate": 6.359 },
                # {"run_number": 14835, "duration": 11713, "OK": 66515  , "LOST": 7358   , "real_rate": 6.306 },
                # {"run_number": 14837, "duration": 55751, "OK": 294175 , "LOST": 31034  , "real_rate": 5.833 },
                # {"run_number": 14838, "duration": 87854, "OK": 431766 , "LOST": 42219  , "real_rate": 5.395 },
                # {"run_number": 14839, "duration": 84881, "OK": 401004 , "LOST": 37747  , "real_rate": 5.169 },
                # {"run_number": 14840, "duration": 56174, "OK": 260266 , "LOST": 23627  , "real_rate": 5.054 }  
            ]

### Reconstructed

In [3]:
RECO_DATA = {run["run_number"]: crudo.dm.load_run_data(run, trigger=None)[run["run_number"]] for run in RUNS_INFO}

/DST/Events: Run 14739 successfully loaded with data shape: (403582, 26)


# Selection

In [ ]:
def selection_criteria(group):
        """
        Selection criteria applied to each group (event-level).
        
        Parameters:
            group (pd.DataFrame): Grouped data corresponding to an individual event.
        
        Returns:
            bool: True if the group meets the selection criteria, otherwise False.
        """
        return (
                    # Multiplicity
                      (group['nS1'].sum() == 1)
                    & (group['nS2'].sum() == 1)
            
                    # S1
                    & (group['S1w'].sum() < 1250)     # 1.25 µs
                    & (group['S1t'].sum() > 150e3)    # 150 µs
            
                    # S2
                    # & (10 < group['S2w'].sum() < 250)   # Primary values [50, 250] µs
                    & (group['S2t'].sum() < 1650e3)   # 1650 µs
            
                    # 214Po-like events: this cut is waveform-based, none correction needed
                    & (group['S1h'].sum() >= 0.17 * group['S1e'].sum() - 56)    # Reject (>=) or select (<)
        )

In [5]:
%%time
sel_data = {run["run_number"]: crudo.filter_run_data(run, raw_data, selection_criteria)[run["run_number"]] for run in runs_info}

Run 14739 filtered successfully. Data shape: (96084, 26)
CPU times: user 2min 47s, sys: 134 ms, total: 2min 47s
Wall time: 2min 48s


# Dataframes Storage

In [6]:
sel_data

{14739:           event          time  s1_peak  s2_peak  nS1  nS2    S1w         S1h  \
 11          331  1.737644e+09        0        0    1    1  600.0  262.389038   
 16          716  1.737644e+09        0        0    1    1  925.0  195.171722   
 18          842  1.737644e+09        0        0    1    1  625.0  211.819778   
 19          849  1.737644e+09        0        0    1    1  625.0  206.751801   
 22         1038  1.737644e+09        0        0    1    1  825.0  216.178925   
 ...         ...           ...      ...      ...  ...  ...    ...         ...   
 403572  2576349  1.737731e+09        0        0    1    1  600.0  269.677124   
 403575  2576384  1.737731e+09        0        0    1    1  700.0  206.194489   
 403576  2576405  1.737731e+09        0        0    1    1  725.0  258.907990   
 403578  2576426  1.737731e+09        0        0    1    1  725.0  362.998657   
 403580  2576587  1.737731e+09        0        0    1    1  550.0  240.844345   
 
                 S1

In [ ]:
# Dataframe name
df_name = 'run_14739'   # 'alpha_runs' (old version), 'Rn_background' (new version), 'run_14739'

# Choose which data to store: sel_data or raw_data
data = sel_data

# Store the selected data
with open(f"/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Rn_analysis/pkl/{df_name}.pkl", "wb") as file:
    pickle.dump(data, file)
    
print('Listo mi pana, péguese un análisis sabroso!')

Listo mi pana, péguese un análisis sabroso!


#### Or do you wanna merge two different dataframes into one?

In [ ]:
merged_data = crudo.merge_dfs(file1="/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Rn_analysis/pkl/temp_runs.pkl", 
                              file2="/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Rn_analysis/pkl/extra_runs.pkl", 
                              output_file="/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Rn_analysis/pkl/alpha_runs.pkl")

Stored merged data in: /data_extra2/ccortesp/NEXT-100/Xe_cmmssnng/data/alpha_runs.pkl
